# Replay Buffer Storage in Continual Learning of Vision-Language Models### Thesis progress notebook: feature replay, REMIND, and what remains**Author:** Alexie Linardatos  **Setting:** Multi-domain Task-Incremental Learning (MTIL), CLIP ViT-B/16This notebook documents the second phase of the thesis. The first phase produced ExRD(ZSCL plus a replay-teacher distillation term), which is written up separately. This phase asks anarrower question:> **What does the replay buffer actually need to contain, and what does it cost?**Contents:1. Background: the setting and where ExRD left off2. The gradient-routing analysis that motivated feature replay3. Feature replay (variant v0): the method and the buffer changes it required4. Representation drift, and a negative result on drift correction5. REMIND, and why it was the right paper to build on6. Why REMIND does not transfer to CLIP: five mismatches7. Our adaptations, and the information-theoretic limit we reached8. All results to date9. Summary of contributions10. Open question: is this a master's thesis?

## 1. Background: the setting, and where ExRD left off### 1.1 MTILMulti-domain Task-Incremental Learning trains a single CLIP model sequentially on 11 datasetsdrawn from very different domains:> Aircraft, Caltech101, CIFAR100, DTD, EuroSAT, Flowers, Food, MNIST, OxfordPet,> StanfordCars, SUN397The benchmark is unusual among continual-learning settings because it measures two things thatpull against each other:| Metric | Definition | What it measures ||---|---|---|| Last | mean accuracy over the 11 trained tasks after the final task | resistance to forgetting || Avg | mean of the whole 11 by 11 task-accuracy matrix | overall quality across the run || Transfer | accuracy on tasks not yet trained (upper triangle) | retention of CLIP's zero-shot ability |Plain fine-tuning wins Last and destroys Transfer. A frozen model does the reverse. The useful worklies between those two failure modes.### 1.2 ExRD, phase one of the thesisZSCL (Zheng et al., ICCV 2023) prevents zero-shot degradation with two distillation terms computedagainst the original pre-trained CLIP on an unlabelled reference corpus.ExRD adds a replay-teacher distillation term, $L_{RD}$, which distils against the previous task'smodel rather than the original CLIP, evaluated on replayed exemplars. ZSCL anchors the model towhere it started; $L_{RD}$ anchors it to where it just was. The two are complementary: onepreserves zero-shot generality, the other preserves recently acquired task knowledge.The full objective:$$L = \underbrace{L_{CE}}_{\text{current task}}  + \underbrace{L_{zscl}}_{\text{vs. pre-trained CLIP}}  + \underbrace{L_{RD}}_{\text{vs. previous-task model}}  + \underbrace{L_{rep}}_{\text{replay CE}}  + \underbrace{L_{2}}_{\text{parameter anchor}}$$ExRD reaches Transfer 69.18, Avg 77.17, Last 86.17, above ZSCL on all three.### 1.3 The cost that is not reportedExRD's replay buffer holds 11,000 preprocessed images, each a 224 by 224 by 3 float32 tensor.| Quantity | Value ||---|---|| Bytes per exemplar | 224 x 224 x 3 x 4 = 602,112 B (602.1 kB) || Exemplars | 11,000 || **Buffer total** | **6.62 GB** |6.62 GB of stored images to train a 150M-parameter model. No MTIL paper reports this number.That observation is the starting point for phase two.

## 2. The gradient-routing analysisBefore trying to shrink the buffer we asked what the buffer is for, specifically which parameterseach loss term updates. CLIP has two towers, image and text, and the answer is asymmetric in a waythat does not appear to have been noted before.Reading the implementation (src/models/training.py, phase3/losses_phase3.py):- $L_{zscl}$ computes the teacher's text embeddings under torch.no_grad(). Gradient reaches the  image encoder only.- $L_{RD}$ does the same. Gradient reaches the image encoder only.- $L_{2}$ is a parameter-space anchor with no data-dependent routing.- $L_{rep}$, the replay cross-entropy, recomputes class-name text embeddings with the live model.  It is therefore the only term in the objective that sends gradient into the text encoder for  previously seen classes.One subtlety worth recording: the --text_loss flag transposes the logits, changing the directionof the softmax. It does not route gradient into the text encoder. This is easy to misread.| Loss term | Image encoder | Text encoder (old classes) ||---|---|---|| $L_{CE}$ (current task) | yes | yes || $L_{zscl}$ | yes | no || $L_{RD}$ | yes | no || $L_{2}$ | no | no || $L_{rep}$ (replay) | yes | yes |Excluding the current-task cross-entropy, the image encoder is constrained by **three** terms andthe text encoder for old classes by **one**, which is replay.### 2.1 The hypothesis this suggestsReplay's contribution decomposes into two halves:- an image-side gradient, largely redundant with $L_{zscl}$ and $L_{RD}$;- a text-side gradient, which nothing else supplies.Storing full images is how the image-side gradient is obtained. If that half is redundant, theimages are unnecessary, and a stored 512-d embedding is sufficient to produce the text-sidegradient.This is the hypothesis feature replay tests. No prior latent-replay work is framed in terms ofwhich tower the replay gradient reaches, because prior work is single-tower.

## 3. Feature replay (variant v0)### 3.1 The methodAt the end of task $t$, encode that task's exemplars once with the current image encoder and storethe resulting 512-d embeddings in fp16. Discard the images.Under pixel replay the replay loss is$$\text{logits} = s \cdot f_\theta(x)\, g_\phi(c)^\top$$where $f_\theta$ is the image encoder, $g_\phi$ the text encoder, $x$ a stored image and $c$ theclass names. Both $\theta$ and $\phi$ receive gradient.Under feature replay, $f_\theta(x)$ is replaced by a stored constant $\hat{z}$:$$\text{logits} = s \cdot \hat{z}\, g_\phi(c)^\top,\qquad\frac{\partial}{\partial \theta} = 0 .$$Only the text encoder is updated, which is the half identified as unique in section 2.### 3.2 Storage| Method | Bytes per exemplar | Buffer at 11,000 ||---|---|---|| Pixel replay (ExRD) | 602.11 kB | 6,623 MB || Feature replay (v0) | 1.02 kB | 11.3 MB |Compression ratio: **588x**.### 3.3 What had to change in the replay bufferThe existing ReplayBuffer stored image tensors and handed them to a dataloader that appliedaugmentation. Feature replay required a parallel implementation(src/feature_replay_buffer.py) with four structural differences:| | Pixel buffer | Feature buffer ||---|---|---|| Stored item | image tensor | 512-d fp16 embedding || When written | at exemplar selection | at the end of the task, after training, in one pass || Augmentation | fresh RandomResizedCrop every epoch | impossible, the vector is fixed || Refreshable | not applicable, pixels do not go stale | no, re-encoding needs the discarded images |Two consequences deserve emphasis, because both became findings.**Augmentation is lost.** A stored vector presents the identical training signal at every replaystep. Pixel replay never repeats itself. This was flagged in advance as the most likely reasonfeature replay would underperform.**$L_{RD}$ loses its input.** The replay-teacher distillation term pushes the same image throughthe student and the frozen teacher. With no images in the buffer it has nothing to consume, and wasswitched to current-task images (rd_source=current). A rebuttal control measured this substitutionas costing about 0.06 Transfer, but its effect on Last was never measured. That remains an openconfound in the results below.The design was implemented as a flag on the existing trainer,--replay_storage {pixel,feature,remind}, so pixel runs remain bit-identical to ExRD and thecomparison is controlled.

## 4. Representation drift### 4.1 The mechanismA stored embedding describes the encoder as it was at the moment of storage. The encoder keepstraining. After $k$ further tasks the stored vector for task $t$ describes a function that no longerexists, and it cannot be refreshed because the images are gone.The prediction is that damage scales with how long a task has been sitting in the buffer. That iswhat we observe.### 4.2 The penalty is concentrated in one taskFinal-row (Last) accuracies, 11-task MTIL, single seed:| Task | Position | ExRD (pixel) | featrep v0 | Delta ||---|---|---|---|---|| **Aircraft** | 0 | 54.52 | 44.55 | **-9.97** || Caltech101 | 1 | 94.07 | 93.26 | -0.81 || CIFAR100 | 2 | 81.98 | 79.74 | -2.24 || DTD | 3 | 76.06 | 74.73 | -1.33 || EuroSAT | 4 | 94.61 | 95.02 | +0.41 || Flowers | 5 | 97.02 | 96.96 | -0.06 || Food | 6 | 90.75 | 90.18 | -0.57 || MNIST | 7 | 98.05 | 98.45 | +0.40 || OxfordPet | 8 | 94.90 | 94.74 | -0.16 || StanfordCars | 9 | 85.54 | 84.63 | -0.91 || SUN397 | 10 | 80.36 | 80.45 | +0.09 || **Mean (Last)** | | **86.17** | **84.79** | **-1.38** |Aircraft accounts for 65.8% of the total loss. Aircraft and CIFAR100 together account for 80.6%.This is the central empirical finding of phase two. The penalty is not diffuse. Aircraft is task 0,stored first, exposed to ten subsequent task boundaries, and the hardest fine-grained dataset in thebenchmark. Eight of eleven tasks are within noise of pixel replay, and two are slightly better.Drift is real and concentrated at the front of the sequence, as the mechanism predicts.### 4.3 The attempted fix, and a negative resultWe implemented a ladder of stale-feature adaptation methods (src/feature_adaptation.py). At eachtask boundary, before adding the new task, stored vectors are corrected toward where the currentencoder would place them. Estimating that correction requires anchors, meaning quantities observableunder both the old and the new encoder:| Anchor type | Source | Note ||---|---|---|| image | probe images pushed through both encoders | standard (Yu CVPR 2020, Iscen ECCV 2020) || text | old classes' text embeddings | exactly recomputable at any time, with no stored image || both | union of the two | |with four estimators: sdc (kernel drift, Yu CVPR 2020), lp (label propagation, Zhang ECCV 2020),and linear / mlp (learned maps, Iscen ECCV 2020).The text-anchor arm was intended as the novel contribution. An old class's text embedding isrecomputable exactly, unlike an image anchor which must be stored or synthesised, so it provides ameasured drift vector co-located with that class's stale image features.It did not work. Full results are in section 8. In summary:- Aggregate Last moved by at most 0.24 points across all four variants.- On Aircraft, where drift damage actually lives, the ordering was image > both > text, with text  approximately equal to no adaptation at all. Text anchors recovered about 0.3 points; image  anchors recovered about 2.2 points.The likely cause is CLIP's modality gap. Text and image embeddings occupy separated regions of theshared space, so a drift vector measured at the text embedding does not transfer to image features.A constant offset would cancel in a difference, so the offset is evidently not constant.This is reported as a negative result. It is also partially anticipated by SeGP-CL(arXiv 2603.12055, 2026), which uses old-class text embeddings as targets for constructingadversarial anchors and adds a visual-prototype term specifically to compensate the modality gap.### 4.4 A measurement problemTask 0 accuracy must be identical across runs, because the buffer is empty and storage mode cannotmatter yet.| Run | --replay_storage | Aircraft (task 0) ||---|---|---|| ExRD (pixel) | pixel | 52.48 || REMIND | remind | 52.48 || featrep v0 | feature | 52.42 || featrep lp_both | feature | 52.60 || featrep lp_text | feature | 53.23 || featrep sdc_image | feature | 52.03 |ExRD and REMIND reproduce each other exactly. Every --replay_storage feature run diverges, with aspread of 1.20 points, and the divergence tracks adaptation flags that are not read until task 1.No --seed differs between these scripts, so the pipeline is deterministic and this is a bug in thefeature-storage code path rather than hardware nondeterminism.Why it matters: the four feature-replay runs diverge from task 0, before adaptation can act. Their0.24 point spread on Last is smaller than this artefact, so the adaptation ladder's null result issuggestive but not cleanly measured, and should be re-run once the bug is fixed.Why it is useful: because the pipeline is otherwise deterministic, single-seed comparisons are validin this codebase and no multi-seed sweep is required.

## 5. REMIND, and why it was the right paper to build onFeature replay stores the final embedding, which is a dead constant, so no gradient can flow intothe image tower at all. REMIND (Hayes et al., ECCV 2020) proposes a middle path.### 5.1 The three ideas1. **Store mid-network activations rather than final embeddings.** Tap the network partway up. The   replayed sample still flows through the layers above it, so it carries a real gradient.2. **Compress with product quantization.** Split each $D$-dimensional vector into $m$ contiguous   subvectors, learn a 256-entry codebook per subspace by k-means, and store one byte per subspace.   A $D = 768$ vector at $m = 32$ becomes 32 bytes, a 96x compression.3. **Freeze everything below the tap after the first task.** The layers that produced the stored   activations never change again, so the activations cannot go stale. Drift is eliminated by   construction rather than corrected after the fact.Point 3 is a direct structural answer to the problem described in section 4.### 5.2 Why REMIND was the right paper, and effectively the only oneThe requirement that emerged from section 4 has two parts, and they are jointly restrictive:> **(a)** prevent drift rather than correct it, since correction was measured and found weak; and> **(b)** keep the replayed sample on a live gradient path, since a dead constant gives up the image> tower entirely.Surveying what else is available, every alternative fails one of the two:| Family | Representative work | Satisfies (a)? | Satisfies (b)? ||---|---|---|---|| Drift compensation | SDC (Yu CVPR 2020), Iscen ECCV 2020, A2LP (Zhang ECCV 2020) | no, corrects after the fact | no, final-layer features || Generative latent replay | CGIL (BMVC 2024 oral) | partly, samples are re-drawn but the VAE itself drifts | no, final-layer embeddings || Generative pixel replay | LoRA-Loop (2025), DGR-style methods | yes, regenerates images | yes, but reintroduces a full pixel pipeline plus a generator to train || Rehearsal-free | AFA (2025), DIKI, BCL | not applicable, no buffer | not applicable || Compressed activation replay | **REMIND (ECCV 2020)**, ACAE-REMIND, SHARP | **yes, by freezing below the tap** | **yes, via the mid-network tap** |Three observations follow.First, the drift-compensation family is what the adaptation ladder already implements, and section4.3 shows it does not recover the loss. Having measured that, moving to prevention was the correctnext step rather than trying a fourth estimator.Second, the rehearsal-free family answers a different question. Those methods do not have a buffer,so they cannot inform a study about what a buffer should contain. They are the right competitivebaselines (section 8) but they are not something to build on here.Third, within the compressed-activation family, ACAE-REMIND and SHARP are both refinements ofREMIND rather than independent alternatives. ACAE-REMIND swaps product quantization for anauxiliary-classifier autoencoder; SHARP adds sparsity and replays only the most recent classes.Both inherit REMIND's split-and-freeze structure. Implementing REMIND therefore implements thecommon core of the entire family, and the two refinements become follow-up ablations rather thanseparate projects.The conclusion is that REMIND is not one option among many. It is the canonical member of the onlyfamily that satisfies both requirements, and porting it is what makes the resulting comparisonlegible to readers who know this literature.### 5.3 Why CNNs make REMIND workREMIND was built on a ResNet-18 for class-incremental ImageNet. Three properties of that settingmake it effective:- Post-ReLU feature maps are sparse and highly redundant, so effective dimensionality is far below  nominal and aggressive quantization is close to lossless.- A CNN has a genuine low-to-high hierarchy. Early layers learn generic edge and texture detectors,  so freezing them costs little and the split point is principled.- There is a single domain. The codebook is fitted once on the first task and reused, which is sound  when every later task is drawn from the same distribution.Each of those three is false for CLIP on MTIL.

## 6. Why REMIND does not transfer to CLIPWe ported REMIND to CLIP ViT-B/16, tapping at block 6 of 12 (src/remind_buffer.py), and found fivemismatches. The first four are engineering problems. The fifth is information-theoretic.### 6.1 Mismatch 1: one codebook, eleven domainsThe original implementation fits the codebook on task 0 only, on the following reasoning:> Refitting later would change what every previously stored code means, so it is deliberately a> one-time event.That is correct for a shared codebook and harmless on single-domain ImageNet. On MTIL it means acodebook fitted on Aircraft is used to quantize MNIST, EuroSAT and DTD.Measured reconstruction error on the first 11-task run was 0.5804, and that figure was measured onAircraft's own tokens, meaning in-distribution.### 6.2 Mismatch 2: outlier channels in the residual streamProduct quantization splits a vector into contiguous subvectors and implicitly assumes variance isspread evenly across dimensions. Transformer residual streams violate this: a small number ofchannels carry magnitudes an order of magnitude above the rest. Those channels concentrate into afew subspaces which 256 centroids cannot cover, while the remaining subspaces are starved.This calls for per-channel standardisation across the token population, not per-token LayerNorm. LNrescales each token across $D$ but leaves the relative scale between channels untouched, and thebetween-channel scale is the part that causes the problem.### 6.3 Mismatch 3: a ViT has no natural seamREMIND splits a ResNet at layer 4.0 because CNNs have a real hierarchy. Every ViT block is the samefull-width residual update over all tokens, with content-dependent attention at every depth. Thereis no principled split point. Block 6 was a guess and remains unswept.### 6.4 Mismatch 4: the classifier is a whole encoder, and it driftsREMIND replays into a linear classifier head that is trained continuously, so there is no stalenesson the classifier side. CLIP has no head: classification is $\hat{z} \cdot g_\phi(c)^\top$ againstclass-name embeddings produced by an encoder that keeps moving, constrained only by $L_{rep}$ asshown in section 2.A correct REMIND port therefore inherits a text-side drift problem that REMIND never had toconsider. This appears to be a new problem statement for latent replay in vision-language models.### 6.5 Mismatch 5: the bit budget cannot buy the fidelityEach subvector holds $D/m$ dimensions coded in 8 bits, so the rate per dimension is $R = 8m/D$. Fora Gaussian source under squared error, $D(R) = \sigma^2 2^{-2R}$, so the relative L2 error floor is$2^{-R}$.| $m$ | Dims per subvector | Bits per dim | Error floor | kB per exemplar | Buffer at 11k ||---|---|---|---|---|---|| 32 | 24 | 0.33 | 0.794 | 6.3 | 69 MB || 64 | 12 | 0.67 | 0.630 | 12.6 | 139 MB || 96 | 8 | 1.00 | 0.500 | 18.9 | 208 MB || 128 | 6 | 1.33 | 0.397 | 25.2 | 277 MB || 192 | 4 | 2.00 | 0.250 | 37.8 | 416 MB || 256 | 3 | 2.67 | 0.157 | 50.4 | 555 MB |Observed quantised-space errors at $m = 32$ were 0.765, 0.740 and 0.487 on DTD, EuroSAT and MNIST,against a predicted floor of 0.794. The codebook is already operating at the theoretical limit forits bit budget. The values sit slightly below the isotropic floor because real data retains somestructure to exploit.Two consequences follow:- OPQ, meaning a learned rotation, will not rescue this. A rotation rebalances which subspaces carry  variance but cannot beat the rate bound.- Whitening pushes toward the bound, because making data isotropic removes exactly the structure  product quantization was exploiting. It fixes load balancing across subspaces and costs  compressibility, and the two effects partially cancel.The deeper point is that REMIND's compression ratio was never transferable. It relied on CNN featuremaps being sparse and low-rank. ViT residual streams are much closer to full-rank, so the same bitbudget buys far less fidelity. Reaching roughly 0.15 relative error would require about $m = 256$,which is 50 kB per exemplar and a 555 MB buffer.

## 7. Our adaptationsTwo changes are implemented and verified running (src/remind_buffer.py,src/product_quantizer.py), both enabled by default with ablation flags.### 7.1 Per-task codebooks (--remind_shared_codebook disables)Each task fits and stores its own codebook alongside its codes. Because every task decodes with thecodebook it was encoded by, a later fit cannot change the meaning of an earlier code, so theconstraint that motivated REMIND's fit-once policy no longer applies.### 7.2 Per-channel whitening (--remind_no_whiten disables)Each task's token population is standardised per dimension before quantization and un-standardisedon decode. Statistics are stored at fp16 and rounded before use, so the forward transform is theexact inverse of what decode applies.### 7.3 Storage accountingnbytes() now includes codebooks and statistics, not only codes:| Component | Size ||---|---|| Codes (11,000 x 197 tokens x 32 B) | 69.3 MB || Codebooks, one per task (11 x 393 kB) | 4.3 MB || Whitening statistics | 33.8 kB || **Total** | **73.7 MB** |Overhead of per-task codebooks is 6.3%. The result is 90x smaller than the 6.62 GB pixel buffer.### 7.4 What the measurement saidSmoke test with both adaptations active, on DTD, EuroSAT and MNIST:| Task | Original space | Quantised space ||---|---|---|| DTD | 0.714 | 0.765 || EuroSAT | 0.641 | 0.740 || MNIST | 0.371 | 0.487 |Three separate codebook fits confirm per-task codebooks are active. The error ordering tracks domaincomplexity, with MNIST far more compressible than DTD, which is itself evidence that mismatch 6.1was real.These figures are not directly comparable to the 0.5804 baseline, because the datasets differ and a200-exemplar smoke budget fits the codebook on roughly 39k token vectors against the full run's 200kcap. Section 6.5 explains why no amount of codebook improvement reaches a low number at this bitbudget.

## 8. All results to dateSingle seed, 11-task MTIL Order-I, CLIP ViT-B/16.### 8.1 Our runs| Run | Avg | Last | T[IN] | T[UT] | Storage | Note ||---|---|---|---|---|---|---|| ExRD / v4 (pixel replay) | 77.17 | 86.17 | 68.37 | 69.18 | 6.6 GB | phase-one baseline || featrep v0 (pure) | 76.08 | 84.79 | 68.12 | 68.81 | 11 MB | no adaptation || featrep lp_both | 76.23 | 84.89 | 67.95 | 68.99 | 11 MB | label propagation, image and text anchors || featrep lp_text | 76.06 | 84.73 | 68.05 | 68.78 | 11 MB | label propagation, text anchors || featrep sdc_image | 76.20 | 84.65 | 67.90 | 68.88 | 11 MB | kernel drift, image anchors || REMIND l6 m32 | 76.00 | 84.62 | 68.55 | 68.88 | 69 MB | shared codebook, no whitening |### 8.2 External baselines| Method | Avg | Last | Transfer | Buffer | Note ||---|---|---|---|---|---|| ZSCL (ICCV 2023) | 75.4 | 83.6 | 68.1 | not reported | reference-data distillation || DIKI | 76.3 | 85.1 | 68.3 | none | rehearsal-free || BCL | 76.7 | 85.0 | 68.9 | none | rehearsal-free || AFA (2025) | 78.5 | 87.2 | 70.3 | none | MoE adapters, zero exemplars |Column legend: **T[IN]** is the mean accuracy on a held-out ImageNet column, the convention usedinternally by this project. **T[UT]** is the upper triangle of the 11 trained tasks, the conventionused by ZSCL, DIKI, BCL and AFA. External comparisons use T[UT] throughout.### 8.3 Storage against accuracy| Method | Buffer | Last ||---|---|---|| ExRD (pixel) | 6.62 GB | 86.17 || REMIND m32 | 69 MB | 84.62 || featrep v0 | 11 MB | 84.79 || AFA (rehearsal-free) | 0 B | 87.20 |### 8.4 Reading the results**The storage claim holds.** 588x less memory for 1.38 points of Last, with eight of eleven tasksunaffected.**The drift-adaptation claim fails.** All four feature-replay variants lie within 0.24 points ofeach other on Last, which is less than the 1.20 point task-0 artefact described in section 4.4.Text anchors did not beat image anchors.**REMIND replaces drift correction with drift prevention, and prevention works better.** Aircraftrecovers from 44.55 to 49.02. The freeze costs plasticity elsewhere, with StanfordCars down 2.70 andEuroSAT down 2.02, so the mean does not move.**Fidelity appears not to be the binding constraint.** REMIND matched feature replay while replayingactivations corrupted at 58% relative error. If fidelity mattered, that could not have happened.**The competitive picture is unflattering.** AFA reaches Last 87.2 and Transfer 70.3 with zerostored exemplars. A storage-efficiency argument cannot be won against zero. Our comparison is faironly within replay-based methods, and against a method that additionally infers task identity atinference through a routing mechanism, which is a materially easier setting than single-modelfine-tuning with no task ID.

## 9. Summary of the thesis so far### Phase one: ExRDAdded $L_{RD}$, a replay-teacher distillation term against the previous-task model, to ZSCL'sdistillation against the original CLIP. Establishes Transfer 69.18, Avg 77.17, Last 86.17, aboveZSCL on all three metrics.### Phase two: what the buffer needs to contain| | Contribution | Status ||---|---|---|| 1 | Gradient-routing analysis: $L_{rep}$ is the only term updating the text encoder for old classes, while $L_{zscl}$ and $L_{RD}$ reach the image encoder only | verified in code, not found in the literature || 2 | Feature replay for MTIL: 588x storage reduction for 1.38 points of Last | implemented, measured || 3 | Per-task drift decomposition: the penalty is about 66% one task, monotone in task age | measured || 4 | Drift-adaptation ladder including text anchors | negative result || 5 | REMIND ported to a ViT and CLIP backbone | no prior port found || 6 | Five documented CLIP mismatches, including the rate-distortion limit | analysed, partly fixed || 7 | Per-task codebooks and per-channel whitening | implemented, running |### What appears unclaimed in the literature- Latent replay evaluated on MTIL with zero-shot Transfer as a metric. CGIL (BMVC 2024), REMIND and  ACAE-REMIND are all class-incremental and none measure zero-shot retention.- Storage footprint reported as an axis on MTIL. ZSCL, BCL, DIKI and AFA report none.- The gradient-routing asymmetry between CLIP's two towers under this objective.- REMIND's compression ratio failing on transformers for a rate-distortion reason.### What is not novel, and should be stated as such- Feature and latent replay as a concept: REMIND (ECCV 2020), ACAE-REMIND (PRL 2021), SHARP (2023).- Latent replay for CLIP: CGIL (BMVC 2024 oral) does generative latent replay in CLIP's embedding  space.- Text embeddings in drift compensation: SeGP-CL (arXiv 2603.12055, 2026) is adjacent, and our  version does not outperform image anchors in any case.

## 10. Open question: is this a master's thesis?Stating the position honestly, for discussion.### The case that it is- A well-posed question, namely what a replay buffer must contain and what it costs, asked in a  setting where it has not been asked.- A mechanism rather than only a number: drift damage is monotone in task age and we can show it.- Two interventions that behave as the mechanism predicts. Post-hoc correction is weak; structural  prevention is stronger but costs plasticity.- Honest negative results with explanations, namely the modality gap and the rate-distortion bound.- It builds directly on phase one, which is what a two-chapter thesis wants.### The case that it is not- No state of the art. AFA beats us on all three metrics with zero exemplars.- The headline novelty claim failed, since text anchors did not beat image anchors.- The storage axis is undercut by rehearsal-free methods that use no buffer at all.- One result is confounded by a bug (section 4.4), and one design concession, the $L_{RD}$  relocation, was never measured on Last.### Specific questions for supervisors1. Is characterisation, mechanism and an honest negative result sufficient, or does the thesis need   a method that wins a number?2. Is the storage axis defensible given rehearsal-free state of the art? Our position is that the   comparison is fair within replay methods, and that AFA's routing mechanism, which infers task   identity at test time, is a different and easier problem setting. Is that framing acceptable?3. Is the gradient-routing analysis in section 2 strong enough to be the primary contribution, with   the empirical work as its validation?4. Is it worth pursuing the one remaining untested idea, depth-stratified replay: storing older   tasks at a deeper layer so fewer trainable blocks sit above them, giving drift protection   proportional to exposure without a global freeze. It costs nothing in storage, since token count   is constant across ViT depth, and it sacrifices only the image-side gradient that section 2   argues is redundant. Estimated at about one day of implementation plus three runs.### Minimum remaining work| | Task | Cost ||---|---|---|| 1 | Fix the task-0 divergence bug (section 4.4) | hours, no GPU || 2 | Measure the $L_{RD}$ relocation on Last, the missing C1b control | 1 run || 3 | Product quantization sweep over $m \in \{32, 64, 128\}$ to test whether fidelity matters | 3 runs, parallel || 4 | Re-run the adaptation ladder after the bug fix | 2 to 3 runs || 5 | Optional: depth-stratified replay | about 1 day plus 3 runs |Roughly 4 to 12 GPU-days, parallelisable, plus writing.The single most informative experiment is number 3. If Last is flat across a 2x range ofreconstruction error, that demonstrates replay fidelity is irrelevant under this objective, whichfollows directly from the gradient-routing analysis in section 2 and would tie the thesis togetherwith a result rather than a caveat.